# Structured output and repair: JSON, schemas, validate-and-repair

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 10 §10.4 · type: walkthrough*

*One-line promise:* get **machine-consumable** output reliably — escalate from prompted JSON to schema-enforced output, then wrap it in a Pydantic **validate-and-repair** loop.

## 🧠 Why this matters

Most production LLM calls don't end at a human — they end at a **parser**. The moment downstream code consumes the output, *"usually valid JSON"* is a bug factory: the one call in fifty that returns prose, a trailing comma, or `"urgency": "high"` instead of an int will take down a job at 3 a.m. The guarantee has to be **engineered**, in increasing strength, and then *validated at the boundary* — because a schema constrains *shape*, not *sense*.

## Objectives & prereqs

**By the end you can:**
- Walk the structured-output ladder: prompted JSON → JSON mode → schema-enforced (strict / the *tool-shaped* trick).
- Define a `Ticket` Pydantic model and `model_validate_json` raw output.
- Build a one-pass **repair loop** that feeds the validation error back and re-validates.

**Prereqs:** notebook `10-01`. Run the setup cell first.

**Cost:** `MOCK=1` (default) returns a canned *malformed-then-fixed* pair so the repair path runs deterministically and free. `MOCK=0` ≈ 1–2 short calls plus one repair.

## Setup

In [ ]:
import json
import os
import random
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # reads a git-ignored .env if present; never hardcode keys

# MOCK=1 (the default) returns canned, realistic responses so this notebook
# runs FREE, OFFLINE, and DETERMINISTICALLY with no API key. Set COMPANION_MOCK=0
# (and ANTHROPIC_API_KEY) to hit the live API once you want to see real outputs.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

# The book's default model. We never call it in MOCK mode; it is here so the
# live path is one flag away and the code shape matches the book.
MODEL = os.getenv("COMPANION_MODEL", "claude-opus-4-8")

random.seed(7)  # any sampling/shuffling below is reproducible

DATA = Path('data')


def load_tickets():
    return json.loads((DATA / 'tickets.json').read_text(encoding='utf-8'))


def load_docs():
    return json.loads((DATA / 'context_docs.json').read_text(encoding='utf-8'))


print('MOCK =', MOCK, '| model =', MODEL)
if not MOCK and not os.getenv('ANTHROPIC_API_KEY'):
    raise SystemExit('MOCK=0 needs ANTHROPIC_API_KEY in your environment / .env')

In [ ]:
from pydantic import BaseModel, ValidationError, field_validator

tickets = load_tickets()
print('pydantic ready; tickets loaded:', len(tickets))

## The ladder: four strengths of "give me JSON"

From weakest (social) to strongest (mechanical):

1. **Prompted JSON.** Describe the shape, show an example. Enforcement is *social*, not mechanical. (Historically you'd also prefill the assistant turn with `{` — but several current models, including the book default `claude-opus-4-8`, **reject** a trailing assistant prefill with a 400, so don't lean on it.)
2. **JSON mode.** A provider flag that guarantees *syntactically valid* JSON — but any shape it likes.
3. **Schema-enforced output.** You supply a JSON Schema and the provider constrains decoding to conform (OpenAI strict mode; tool definitions on either provider).
4. **The tool-shaped trick.** Define a single tool whose *input schema is your output shape* and force the "call" — the arguments are your structured result. The most portable strong option across providers.

### ⚠️ Pitfall: JSON mode guarantees syntax, not shape

The most common mistake is trusting provider *"JSON mode"* to enforce your **shape**. It only guarantees the bytes parse. You can still get `{"category": "billing"}` with no `urgency`, or `urgency` as the string `"3"`. **Validate at the boundary regardless of what the provider promises.**

## Define the contract: a Pydantic model

This `Ticket` model *is* the boundary. `model_validate_json` parses **and** type-checks in one call; anything off-shape raises `ValidationError` instead of silently flowing downstream.

In [ ]:
class Ticket(BaseModel):
    category: str   # billing | bug | feature_request | other
    urgency: int    # 1 (low) .. 4 (critical)
    summary: str

    @field_validator('category')
    @classmethod
    def _known_category(cls, v):
        allowed = {'billing', 'bug', 'feature_request', 'other'}
        if v not in allowed:
            raise ValueError(f'category must be one of {sorted(allowed)}')
        return v

    @field_validator('urgency')
    @classmethod
    def _urgency_range(cls, v):
        if not 1 <= v <= 4:
            raise ValueError('urgency must be 1..4')
        return v


good = Ticket.model_validate_json('{"category": "bug", "urgency": 3, "summary": "Export 500s"}')
print('parsed OK:', good)

### 🔮 Predict

Here's a syntactically valid object: `{"category": "bug", "urgency": 3, "summary": "Customer is double-charged"}`. It validates against `Ticket` — every field is the right type and in range. **Will the triage be correct?** Predict before running the next cell.

In [ ]:
shape_ok = Ticket.model_validate_json(
    '{"category": "bug", "urgency": 3, "summary": "Customer is double-charged"}'
)
print('Validates?  yes:', shape_ok)
print('Correct?    no — a double charge is *billing*, not bug.')
print('Lesson: a schema guarantees SHAPE, not SENSE. Semantic checks live elsewhere')
print('         (the eval suite in 10-03, guardrails in Ch 41).')

## The model call that returns (sometimes broken) JSON

In `MOCK=1`, the first call deliberately returns a *malformed* object (a string urgency and a stray code fence) so the repair path is exercised every run. The second, post-repair call returns the corrected object. In `MOCK=0` this is one real call (and, on failure, one repair call).

In [ ]:
STRUCTURED_SYSTEM = '''\
Classify the ticket. Respond with ONLY a JSON object:
{"category": billing|bug|feature_request|other, "urgency": 1-4, "summary": "..."}
'''

_mock_calls = {'n': 0}


def raw_call(prompt, *, repairing=False):
    """Return the model's raw text. MOCK: first answer is broken; the repair is clean."""
    if MOCK:
        _mock_calls['n'] += 1
        if not repairing:
            # A realistic failure: code fence + urgency as a string.
            return '```json\n{"category": "billing", "urgency": "high", "summary": "Double charge"}\n```'
        return '{"category": "billing", "urgency": 3, "summary": "Double charge; refund duplicate"}'
    from anthropic import Anthropic
    client = Anthropic()
    msg = client.messages.create(model=MODEL, max_tokens=200, system=STRUCTURED_SYSTEM,
                                  messages=[{'role': 'user', 'content': prompt}])
    return msg.content[0].text


ticket = next(t for t in tickets if t['id'] == 'T-001')
raw = raw_call(f"<ticket>{ticket['text']}</ticket>")
print('raw model output:')
print(raw)

### Watch it fail (on purpose)

That output is *almost* JSON. The code fence and the string `"high"` make `model_validate_json` raise. Good — a loud failure at the boundary beats a silent one downstream.

In [ ]:
try:
    Ticket.model_validate_json(raw)
except ValidationError as e:
    print('ValidationError (expected):')
    print(e)

## The validate-and-repair loop

The shape from the book (§10.4): parse; on `ValidationError`, **feed the error back** and ask for a corrected object; validate again. One repair pass resolves the bulk of residual failures cheaply. The capstone wraps *every* structured call in exactly this.

In [ ]:
def parse_ticket(raw, retry, *, max_repairs=1):
    """Validate raw JSON into a Ticket; on failure, one (or more) repair passes."""
    for attempt in range(max_repairs + 1):
        try:
            return Ticket.model_validate_json(_strip_fences(raw))
        except ValidationError as e:
            if attempt == max_repairs:
                raise
            raw = retry(f'Fix this JSON to satisfy the schema. Error: {e}\n\n{raw}')


def _strip_fences(text):
    t = text.strip()
    if t.startswith('```'):
        t = t.split('```')[1]
        t = t[4:] if t.lower().startswith('json') else t
    return t.strip()


def retry(prompt):
    return raw_call(prompt, repairing=True)


ticket_obj = parse_ticket(raw, retry)
print('repaired & validated:', ticket_obj)
if MOCK:
    print(f"(mock model calls used: {_mock_calls['n']} — one bad, one repair)")

### 🔧 Build: a reusable structured-call wrapper

Bundle the call + parse + repair into one function. This is the shape the capstone platform exposes everywhere a structured result is needed.

In [ ]:
def structured_triage(ticket_text):
    raw = raw_call(f'<ticket>{ticket_text}</ticket>')
    return parse_ticket(raw, retry)


result = structured_triage('I was charged twice for my subscription.')
print(result.model_dump())
assert isinstance(result, Ticket) and 1 <= result.urgency <= 4
print('guaranteed: a typed, in-range Ticket object downstream code can trust.')

## A note on prefill

Older guides prefill the assistant turn with `{` to *nudge* JSON. Treat prefill as a portable *concept*, not a portable *guarantee*: several current models — the book default `claude-opus-4-8` among them — **reject** a trailing assistant prefill with a 400. Schema-constrained output has superseded prefill for getting format *guaranteed* rather than nudged.

## 🎯 Senior lens

**Validate at the boundary regardless of provider guarantees.** Provider JSON mode, strict schemas, tool-shaped calls — use the strongest your stack offers, *and still* parse into a typed model at the edge, because the failure you didn't guard is the one that pages you. The repair loop is cheap insurance: one extra call on the rare miss, versus a downstream crash. This exact pattern governs tool **inputs** in Ch 12 and every structured call in the Ch 15 capstone — learn it once here.

## Recap

- Most calls end at a **parser**, so engineer the guarantee — don't hope.
- The ladder: prompted JSON → JSON mode → schema-enforced → the **tool-shaped** trick (most portable strong option).
- A schema guarantees **shape, not sense** — always validate at the boundary with a typed model.
- **Validate-and-repair:** parse; on error, feed the error back for one correction; re-validate. Cheap, and it clears most residual failures.
- **Prefill is not a guarantee** — some current models reject it outright.

## Exercises

1. **A nastier failure.** Make `raw_call` (mock branch) return JSON missing the `summary` field. 🔮 Predict the `ValidationError`, then confirm the repair fixes it.
2. **Two repair passes.** Set `max_repairs=2` and craft a mock that needs two rounds. Where would you cap repairs in production, and why not loop forever?
3. **Semantic check.** Add a check (outside Pydantic) that flags the *shape-valid-but-wrong* `bug`/double-charge case from the predict cell. What layer should own that — schema, eval, or guardrail?
4. **Tool-shaped output.** Sketch (no live call needed) an Anthropic `tools=[...]` definition whose input schema is `Ticket`, and describe how `tool_choice` forces the structured result.

In [ ]:
# Exercise 1 — missing-field failure, then repair.


In [ ]:
# Exercise 2 — max_repairs=2 with a two-round mock.


In [ ]:
# Exercise 3 — a semantic check beyond the schema.


## Next

- **Next notebook:** [`10-03-prompts-as-code-registry-and-evals.ipynb`](./10-03-prompts-as-code-registry-and-evals.ipynb) — put these prompts under version control and gate changes on an eval suite (the chapter's 🔧 Build).
- **Blueprint this seeds:** the validate-and-repair reliability pattern feeds [`blueprints/llm-gateway/`](../../blueprints/llm-gateway/) (Ch 11) and [`blueprints/eval-harness/`](../../blueprints/eval-harness/) (Ch 22).
- **Capstone:** the platform-wide structured-call wrapper (Ch 15).